In [7]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR

In [8]:
df = pd.read_csv("Salary.csv")

df.head()

,YearsExperience,Salary
0,1.1,39343
1,1.3,46205
2,1.5,37731
3,2.0,43525
4,2.2,39891


In [9]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (35, 1)
Target Shape: (35,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 28
Testing Samples: 7


In [11]:
rf_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}

rf_search = GridSearchCV(
    RandomForestRegressor(),
    rf_grid,
    cv=5
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("Best Random Forest:")
print(best_rf)

Best Random Forest:
RandomForestRegressor()


In [12]:
svr_grid = {
    'C': [1, 10, 100],
    'kernel': ['linear', 'rbf']
}

svr_search = GridSearchCV(
    SVR(),
    svr_grid,
    cv=5
)

svr_search.fit(X_train, y_train)

best_svr = svr_search.best_estimator_

print("Best SVR:")
print(best_svr)

Best SVR:
SVR(C=100, kernel='linear')


In [14]:
base_models = [
    ('rf', best_rf),
    ('svr', best_svr),
    ('gbr', GradientBoostingRegressor())
]

base_models

[('rf', RandomForestRegressor()),
 ('svr', SVR(C=100, kernel='linear')),
 ('gbr', GradientBoostingRegressor())]

In [15]:
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression()
)

stack_model.fit(X_train, y_train)

print("Stacking Regressor Trained Successfully")

Stacking Regressor Trained Successfully


In [16]:
joblib.dump(
    stack_model,
    "stack_regressor.pkl"
)

print("Model Saved")

Model Saved


In [17]:
sample = X_test.iloc[[0]]

prediction = stack_model.predict(sample)

print("Predicted Salary:", prediction[0])
print("Actual Salary:", y_test.iloc[0])

Predicted Salary: 114490.1824557381
Actual Salary: 116969
